In [15]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc,precision_score,recall_score
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score,f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
import argparse
import warnings
import mlflow
import mlflow.sklearn
from mlflow.models.signature import ModelSignature, infer_signature
from mlflow.types.schema import Schema,ColSpec
import mlflow.xgboost
#import pickle
from pathlib import Path
# ---- Configure warnings and stdout ----
warnings.filterwarnings("ignore", category=UserWarning)
#sys.stdout.reconfigure(encoding='utf-8')
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

#### Get Cleaned Dataset with Feature Scaling

In [16]:
# Function: drop rows with non-numeric values
def drop_non_numeric(df_frame):
    cleaned_df = df_frame.copy()
    for col in cleaned_df.columns:
        # Try to convert column to numeric
        cleaned_df[col] = pd.to_numeric(cleaned_df[col], errors='coerce')
    # Drop rows where conversion failed (NaN introduced)
    cleaned_df = cleaned_df.dropna()
    return cleaned_df

def categorize_chol(val):
    if val < 200:
        return "Normal", 0
    elif 200 <= val <= 239:
        return "Medium", 1
    else:
        return "High", 1
 
def get_cleaned_data():
    # Get current working directory    
    cwd = cwd = os.getcwd()
    # Specify dataset filename
    filename = "dataset_2190_cholesterol.csv"
    file_path = os.path.join(cwd, filename)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ Dataset not found at: {file_path}")
    df = pd.read_csv(file_path)
    null_counts = df.isnull().sum()
    #if null_counts.sum() > 0:
    #    print("\n✅ Null values are present in the dataset.")
    #else:
    #    print("\n❌ No null values found in the dataset.")
    # Apply cleaning
    df_clean = drop_non_numeric(df)
    #print("Cleaned shape:", df_clean.shape)
    df_clean[['chol_category_label', 'chol_category_code']] = df_clean['chol'].apply(
        lambda x: pd.Series(categorize_chol(x))
        )
    # Define columns and target
    columns = ['age', 'sex', 'cp', 'trestbps', 'fbs', 'restecg', 'thalach', 'exang',
           'oldpeak', 'slope', 'ca', 'thal', 'num', 'chol_category_code']
    target = "chol_category_code"
    binary_vars = ['sex', 'fbs', 'exang']
    # Separate features and target
    X = df_clean[columns].drop(columns=[target])
    y = df_clean[target]
    
    # Identify numeric columns to scale (exclude binary + target)
    numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col not in binary_vars]
    
    # Apply StandardScaler
    scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    
    # Store result in df_scale (features + target)
    df_scale = X_scaled.copy()
    df_scale[target] = y
    
    #print("Scaled dataset preview:")
    #print(df_scale.columns)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )
    #print("\nTrain set shape:", X_train.shape, y_train.shape)
    #print("Test set shape:", X_test.shape, y_test.shape)
    return X_train, X_test, y_train, y_test


####  Logistic Model with different experiment name

In [17]:
def logistic_model_1(X_train, X_test, y_train, y_test):
    # Logistic Regression with elasticnet penalty, saga solver
    # 1. saga + elasticnet + l1_ratio=0.1, C=10
    C_value = 10
    L_1_ratio = 0.1
    Solver = 'saga'
    Panality = 'elasticnet'
    model = LogisticRegression(
    penalty=Panality,
    solver=Solver,
    l1_ratio=L_1_ratio,
    C=C_value,
    max_iter=10000
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    y_pred_class = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred_class)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    mse = mean_squared_error(y_test, y_pred_prob)
    mae = mean_absolute_error(y_test, y_pred_prob)

    # Store results in DataFrame
    results_df = pd.DataFrame([{
        "C": C_value,
        "l1_ratio": L_1_ratio,
        'solver': Solver,
        'penalty' : Panality,
        "Accuracy": acc,
        "ROC_AUC": roc_auc,
        "MSE": mse,
        "MAE": mae
    }])
    print("================ Logistice Model ================")
    print(f'Logistice accureacy {acc}')
    return model, results_df

def logistic_model_2(X_train, X_test, y_train, y_test):
    # Logistic Regression with elasticnet penalty, saga solver
    # 1. saga + elasticnet + l1_ratio=0.1, C=10
    C_value = 1
    L_1_ratio = 0.5
    Solver = 'saga'
    Panality = 'elasticnet'
    model = LogisticRegression(
    penalty=Panality,
    solver=Solver,
    l1_ratio=L_1_ratio,
    C=C_value,
    max_iter=10000
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    y_pred_class = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred_class)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    mse = mean_squared_error(y_test, y_pred_prob)
    mae = mean_absolute_error(y_test, y_pred_prob)

    # Store results in DataFrame
    results_df = pd.DataFrame([{
        "C": C_value,
        "l1_ratio": L_1_ratio,
        'solver': Solver,
        'penalty' : Panality,
        "Accuracy": acc,
        "ROC_AUC": roc_auc,
        "MSE": mse,
        "MAE": mae
    }])
    print("================ Logistice Model ================")
    print(f'Logistice accureacy {acc}')
    return model, results_df

def logistic_model_3(X_train, X_test, y_train, y_test):
    # Logistic Regression with elasticnet penalty, saga solver
    # 1. saga + elasticnet + l1_ratio=0.1, C=10
    C_value = 1
    L_1_ratio = 0.9
    Solver = 'saga'
    Panality = 'elasticnet'
    model = LogisticRegression(
    penalty = Panality,
    solver = Solver,
    l1_ratio = L_1_ratio,
    C = C_value,
    max_iter = 10000
    )
    model.fit(X_train, y_train)

    # Predictions
    y_pred_prob = model.predict_proba(X_test)[:, 1]  # probability of class 1
    y_pred_class = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred_class)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    mse = mean_squared_error(y_test, y_pred_prob)
    mae = mean_absolute_error(y_test, y_pred_prob)

    # Store results in DataFrame
    results_df = pd.DataFrame([{
        "C": C_value,
        "l1_ratio": L_1_ratio,
        'solver': Solver,
        'penalty' : Panality,
        "Accuracy": acc,
        "ROC_AUC": roc_auc,
        "MSE": mse,
        "MAE": mae
    }])
    print("================ Logistice Model ================")
    print(f'Logistice accureacy {acc}')
    return model, results_df


In [18]:
def run_experiment(uri='default_path',experiment_name='default_exp', model_name='logistic_model',
                   run_name='default_run', model_func=None, model_args=None, tags=None,
                   input_data = None, output_data = None,input_example=None):
    # Set tracking directory explicitly
    mlflow.set_tracking_uri(uri)
    print("The set tracking uri is ", mlflow.get_tracking_uri())
    # ✅ Use set_experiment to avoid duplicate errors
    exp = mlflow.set_experiment(experiment_name=experiment_name)
    exp_id = exp.experiment_id
    get_exp = mlflow.get_experiment(exp_id)
    print("Name:", get_exp.name)
    print("Experiment_id:", get_exp.experiment_id)
    print("Artifact Location:", get_exp.artifact_location)
    print("Tags:", get_exp.tags)
    print("Lifecycle_stage:", get_exp.lifecycle_stage)
    print("Creation timestamp:", get_exp.creation_time)
    with mlflow.start_run(experiment_id=exp_id, run_name=run_name):
        # Set tags if provided
        if tags:
            mlflow.set_tags(tags)

        # Train and evaluate model
        model, results_df = model_func(**model_args)

        # Log parameters
        for param in ["C", "l1_ratio", "solver",'penalty']:
            if param in results_df.columns:
                mlflow.log_param(param, results_df.loc[0, param])

        # Log metrics
        for metric in ["Accuracy", "ROC_AUC", "MSE", "MAE"]:
            if metric in results_df.columns:
                mlflow.log_metric(metric, results_df.loc[0, metric])

        # Log model
        mlflow.sklearn.log_model(model, name=model_name, serialization_format="skops")
        # Model Signature, Signature Enforcement, and Logging Signatures
        input_schema = Schema([ColSpec(col["type"], col['name']) for col in input_data])
        output_schema = Schema([ColSpec(col['type']) for col in output_data])
        signature = ModelSignature(inputs=input_schema, outputs=output_schema)   
        
        # Log artifacts (optional)
        mlflow.log_artifacts("dataset_2190_cholesterol.csv")
        mlflow.sklearn.log_model(model, name=model_name, signature=signature, input_example=input_example)
        #signature = infer_signature(X_train, model.predict(X_train))
        #mlflow.sklearn.log_model(model, "logistic_model", signature=signature)
        # Print artifact URI
        artifacts_uri = mlflow.get_artifact_uri()
        print("The artifact path is", artifacts_uri)

    # End run explicitly (optional, since 'with' block handles it)
    mlflow.end_run()

    # Show last run info
    run = mlflow.last_active_run()
    if run:
        print("Active run id:", run.info.run_id)
        print("Active run name:", run.info.run_name)
    return


In [19]:
input_data = [
{"name": "age",        "type": "double"},
{"name": "sex",        "type": "double"},
{"name": "cp",         "type": "double"},
{"name": "trestbps",   "type": "double"},
{"name": "fbs",        "type": "double"},
{"name": "restecg",    "type": "double"},
{"name": "thalach",    "type": "double"},
{"name": "exang",      "type": "double"},
{"name": "oldpeak",    "type": "double"},
{"name": "slope",      "type": "double"},
{"name": "ca",         "type": "double"},
{"name": "thal",       "type": "double"},
{"name": "num",        "type": "double"}
]
output_data = [{'type': 'double'}]
input_example = {
    "age": np.array([0.936181, 1.378929, 1.378929, -1.941680, -1.498933]),
    "sex": np.array([1.0, 1.0, 1.0, 1.0, 0.0]),
    "cp": np.array([-2.240629, 0.873880, 0.873880, -0.164289, -1.202459]),
    "trestbps": np.array([0.750380, 1.596266, -0.659431, -0.095506, -0.095506]),
    "fbs": np.array([1.0, 0.0, 0.0, 0.0, 0.0]),
    "restecg": np.array([1.010199, 1.010199, 1.010199, -1.003419, 1.010199]),
    "thalach": np.array([0.017494, -1.816334, -0.899420, 1.633010, 0.978071]),
    "exang": np.array([0.0, 1.0, 1.0, 0.0, 0.0]),
    "oldpeak": np.array([1.068965, 0.381773, 1.326662, 2.099753, 0.295874]),
    "slope": np.array([2.264145, 0.643781, 0.643781, 2.264145, -0.976583]),
    "ca": np.array([-0.721976, 2.478425, 1.411625, -0.721976, -0.721976]),
    "thal": np.array([0.655877, -0.894220, 1.172577, -0.894220, -0.894220]),
    "num": np.array([-0.767668, 0.855089, 0.043711, -0.767668, -0.767668])
    #"chol_category_code": np.array([1.0, 1.0, 1.0, 1.0, 1.0]),
}


In [20]:
def main():
    warnings.filterwarnings("ignore")
    np.random.seed(40)
    # Load data
    X_train, X_test, y_train, y_test = get_cleaned_data()
    # Example usage
    tags = {
    "Work": "Embedded Platform",
    "release.candidate": "RELAY_01",
    "release.version": "1.0.10",
    "dataset": "Cholesterol",
    "experiment.stage": "hyperparameter_tuning",
    "owner": "abhishek",
    "framework": "scikit-learn",
    "model.type": "logistic"
    }
    #uri_path = 'file:D:/ML/MLFlow/MLFlow-6_ModelSignature/mlruns'
    uri_path='http://127.0.0.1:5000'
    run_experiment(uri=uri_path,
                   experiment_name='logistice_signature', 
                   model_name='logistice_reg_signature',
                   run_name='Model_runs', 
                   model_func=logistic_model_1,
                   model_args={
                                "X_train": X_train,
                                "X_test": X_test,
                                "y_train": y_train,
                                "y_test": y_test,        
                                },
                   tags=tags,
                   input_data = input_data, 
                   output_data = output_data,
                   input_example =input_example
                   )

In [21]:
if __name__ == "__main__":
    main()

2026/06/03 21:05:57 INFO mlflow.tracking.fluent: Experiment with name 'logistice_signature' does not exist. Creating a new experiment.


The set tracking uri is  http://127.0.0.1:5000
Name: logistice_signature
Experiment_id: 14
Artifact Location: file:C:/Users/abhis/mlruns/14
Tags: {}
Lifecycle_stage: active
Creation timestamp: 1780500957012
================ Logistice Model ================
Logistice accureacy 0.8333333333333334


2026/06/03 21:06:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


The artifact path is file:C:/Users/abhis/mlruns/14/a77040ba7bd14d89ad61b93e80197be1/artifacts
🏃 View run Model_runs at: http://127.0.0.1:5000/#/experiments/14/runs/a77040ba7bd14d89ad61b93e80197be1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/14
Active run id: a77040ba7bd14d89ad61b93e80197be1
Active run name: Model_runs


#### Where to Find the Signature:
Go to Experiments → select your experiment (e.g., `logistice_signature`). Click on a specific `Run`. In the `Artifacts section`, open the logged model folder (e.g., `logistice_reg_signature`).Inside that folder, MLflow stores: MLmodel file → contains metadata including the `signature schema` (inputs, outputs, and example). `model.pkl` or skops format → the serialized model. `conda.yaml` / requirements.txt → environment dependencies.

![Signature-1](image.png)
![Signature-2](image-1.png)